# Experiment 3.13: SGD-to-Muon Hybrid Fine-Tuning Strategy

## Theoretical Motivation: Two-Phase Optimization in Parameter Space

Deep linear networks define a smooth loss landscape whose geometry is governed by the
**product structure** of weight matrices $W_{\text{eff}} = W_L \cdots W_2 \cdot W_1$.
Gradient descent on this product manifold faces a well-known tension:

1. **Early training (large-gradient regime):** The loss surface is dominated by high-curvature
   directions. Standard SGD with momentum excels here because it follows the steepest descent
   efficiently, making rapid progress along these dominant eigendirections of the Hessian.
   The gradients are large and well-conditioned; there is no need for sophisticated spectral
   manipulation.

2. **Late training (small-gradient, ill-conditioned regime):** As the network approaches a
   minimum, the remaining loss resides in directions with small, poorly-separated singular
   values. SGD slows dramatically because its effective step size scales with gradient magnitude.
   **Muon** (Momentum + Orthogonalization via Newton-Schulz) overcomes this by projecting
   gradients onto the nearest orthogonal matrix, yielding a **spectrally flat update** whose
   singular values are all unity. This acts as an implicit gauge-fixing mechanism in the
   renormalization group (RG) sense: it removes the redundant rescaling degrees of freedom
   between layers, forcing the optimizer to make progress along the physically meaningful
   (gauge-invariant) directions.

### The Hybrid Hypothesis

If SGD is fast-but-shallow and Muon is slow-to-start-but-deep, then a **two-phase strategy**
should capture the best of both worlds:

- **Phase 1 (steps 0 to S-1):** Use SGD+momentum to rapidly reduce loss along the
  high-curvature directions. This is the "coarse-graining" phase in the RG analogy.
- **Phase 2 (steps S to N-1):** Switch to Muon to refine the solution along the
  remaining ill-conditioned directions. This is the "fine-graining" or gauge-fixing phase.

The critical question is: **What is the optimal switch point S?**

## Hypothesis

- S=0 (pure Muon) has best final loss but slower early
- S=200 (pure SGD) has fastest early but worse final
- Optimal S is somewhere in between (maybe S=5-20)

## Methodology

- 4-layer deep linear network, 32x32 weight matrices
- Pre-train with SGD for 500 steps on $W_{\text{target,original}}$
- Modify 20% of target entries $\rightarrow W_{\text{target,modified}}$ (simulating distribution shift)
- Fine-tune for 200 total steps with hybrid: SGD for S steps, then Muon for (200-S) steps
- Sweep S in {0, 5, 10, 20, 50, 100, 200} (S=0 = pure Muon, S=200 = pure SGD)
- 5 random seeds for statistical robustness

## Expected Outcomes

The hypothesis predicts a U-shaped (or monotonic) relationship between S and final loss,
with the optimum at a small but nonzero S value. If confirmed, this demonstrates that
SGD and Muon are **complementary** optimizers operating on different spectral regimes
of the loss landscape, consistent with the RG gauge-fixing interpretation.

## Prior Context

This experiment extends the core mechanism tests for Muon-as-RG-Gauge-Fixing by asking
whether the spectral orthogonalization performed by Muon is most valuable in the late
stages of optimization, where standard gradient methods lose effectiveness due to
ill-conditioning of the effective weight product.

## Measurements

- **Final loss** after 200 fine-tuning steps (solution quality)
- **Steps to reach 50% of initial loss** (convergence speed)
- **Optimal S** that achieves both fast early convergence AND best final loss

## Environment Setup

Import required libraries and configure the computational environment.

All computations use NumPy for transparency: every matrix multiplication, SVD projection,
and gradient computation is explicit and inspectable, with no framework autograd obscuring
the mechanics.

In [ ]:
import numpy as np
import os

---

## Experimental Configuration

### Reproducibility

A fixed global seed ensures identical initial weights, data matrices, and target
perturbations across runs, making results fully reproducible.

In [ ]:
SEED = 42
np.random.seed(SEED)

print(f"Random seed set for reproducibility")


### Network Architecture Parameters

A **4-layer deep linear network** with 32x32 weight matrices. The depth is critical:
with $L=4$ layers the effective weight matrix $W_{\text{eff}} = W_4 W_3 W_2 W_1$ creates a
**quartic** relationship between individual layer parameters and the network output. This
amplifies the condition number of the loss Hessian relative to a single-layer model, making
the late-training ill-conditioning problem that Muon addresses particularly acute.

The batch size of 64 provides $2 \times$ oversampling relative to the 32-dimensional input
space, giving stable gradient estimates without excessive averaging.

In [ ]:
DIM = 32
NUM_LAYERS = 4
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_LAYERS = {NUM_LAYERS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")


### Pre-Training Phase Parameters

The network is first pre-trained with vanilla SGD for 500 steps at lr=0.01 on the
**original** target $W_{\text{target,original}}$. This establishes a well-converged
baseline -- the weights have already settled into a region of parameter space that
encodes the original task. The pre-trained checkpoint is the starting point for all
fine-tuning runs, ensuring a fair comparison across switch points.

In [ ]:
# Pre-training
PRETRAIN_STEPS = 500
PRETRAIN_LR = 0.01

### Fine-Tuning Phase Parameters

Fine-tuning runs for 200 steps total, split between SGD and Muon at the switch point S.

Note the **asymmetric learning rates**: SGD uses lr=0.01 while Muon uses lr=0.005. This is
intentional -- Muon's orthogonalized gradients have unit spectral norm by construction,
so they deliver larger effective updates per step than raw gradients. A lower learning rate
compensates for this amplification and prevents overshooting.

In [ ]:
# Fine-tuning
FINETUNE_STEPS = 200
SGD_FT_LR = 0.01
MUON_FT_LR = 0.005

### Muon Optimizer Parameters

- **Momentum = 0.9**: Standard heavy-ball momentum coefficient, shared by both SGD and Muon
  phases for fair comparison.
- **Newton-Schulz iterations = 5**: The iterative orthogonalization procedure that projects
  each gradient matrix onto the Stiefel manifold $O(n)$. Five iterations typically suffice
  for convergence to machine precision for 32x32 matrices. This is the key operation that
  equalizes all singular values of the update to unity.

In [ ]:
# Muon parameters
MOMENTUM = 0.9
NS_ITERS = 5

### Distribution Shift Simulation

Modifying 20% of the target weight entries simulates a **distribution shift** -- the task
the network was pre-trained on has partially changed. This creates a fine-tuning scenario
where the network must adapt to new information while retaining most of its prior knowledge.
The 20% fraction is chosen to be large enough that fine-tuning is non-trivial, but small
enough that the pre-trained weights provide a meaningful warm start.

In [ ]:
# Target modification
MODIFY_FRAC = 0.20

### Switch Point Sweep Design

The sweep covers the full spectrum from pure Muon (S=0) to pure SGD (S=200), with
intermediate points concentrated at low S values {5, 10, 20} where we hypothesize
the optimum lies, plus larger values {50, 100} to map the degradation curve.

**Interpretation guide:**
- S=0: Muon runs for all 200 steps (pure spectral orthogonalization)
- S=5: SGD warms up for 5 steps, then Muon takes over for 195 steps
- S=200: SGD runs for all 200 steps (pure gradient descent, no orthogonalization)

In [ ]:
# Sweep
SWITCH_POINTS = [0, 5, 10, 20, 50, 100, 200]  # S=0 means pure Muon, S=200 means pure SGD

In [ ]:
NUM_SEEDS = 5

In [ ]:
print("\n--- Full Experimental Configuration Summary ---")
print(f"  Pre-training:  {PRETRAIN_STEPS} steps, SGD lr={PRETRAIN_LR}, momentum={MOMENTUM}")
print(f"  Fine-tuning:   {FINETUNE_STEPS} steps total per run")
print(f"    SGD phase:   lr={SGD_FT_LR}, momentum={MOMENTUM}")
print(f"    Muon phase:  lr={MUON_FT_LR}, momentum={MOMENTUM}, NS iters={NS_ITERS}")
print(f"  Target modification: {int(MODIFY_FRAC*100)}% of entries randomized")
print(f"  Switch points: {SWITCH_POINTS}")
print(f"  Seeds:         {NUM_SEEDS} (base={SEED}, stride=137)")
print(f"  Total runs:    {NUM_SEEDS * len(SWITCH_POINTS)} ({NUM_SEEDS} seeds x {len(SWITCH_POINTS)} switch points)")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


---

## Data Generation

### Input and Target Matrices

The input data $X \in \mathbb{R}^{32 \times 64}$ is drawn i.i.d. from $\mathcal{N}(0, 0.09)$
(scaled by 0.3). The target weight matrix $W_{\text{target}} \in \mathbb{R}^{32 \times 32}$
is drawn from $\mathcal{N}(0, 0.25)$ (scaled by 0.5). These moderate scales prevent
numerical overflow in the 4-layer product while maintaining a non-trivial optimization
landscape.

The target output is $Y = W_{\text{target}} X$, so the network must learn to factor
$W_{\text{target}}$ as a product of four matrices. This factorization is not unique --
the **gauge freedom** $W_{l+1} G \cdot G^{-1} W_l = W_{l+1} W_l$ for any invertible $G$
means infinitely many weight configurations produce identical outputs. This gauge
redundancy is precisely what Muon's orthogonalization addresses.

In [ ]:
# Fixed data
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3
W_target_original = np.random.randn(DIM, DIM) * 0.5

print("\n--- Data Shapes & Statistics ---")
print(f"  X_data: shape={X_data.shape}, mean={X_data.mean():.6f}, std={X_data.std():.6f}")


In [ ]:
# Diagnostic: examine spectral properties of the target
svs_target = np.linalg.svd(W_target_original, compute_uv=False)
print(f"\n--- Target Matrix Spectral Properties ---")
print(f"  W_target_original: shape={W_target_original.shape}")
print(f"  Frobenius norm:    {np.linalg.norm(W_target_original, 'fro'):.4f}")
print(f"  Spectral norm:     {svs_target[0]:.4f}")
print(f"  Smallest SV:       {svs_target[-1]:.4f}")
print(f"  Condition number:  {svs_target[0] / (svs_target[-1] + 1e-12):.2f}")
print(f"  Effective rank:    {(np.sum(svs_target)**2) / (np.sum(svs_target**2) + 1e-12):.2f}")
print(f"\n  The condition number of W_target sets the baseline difficulty.")
print(f"  With L=4 layers, the Hessian condition number scales roughly as kappa^{2*NUM_LAYERS},")
print(f"  so kappa(target)={svs_target[0]/svs_target[-1]:.1f} implies Hessian kappa ~ {(svs_target[0]/svs_target[-1])**(2*NUM_LAYERS):.0f}")

---

## Network Initialization and Utility Functions

### Weight Initialization: Near-Identity Start

Each layer is initialized as $W_l = I + 0.1 \cdot \epsilon$ where $\epsilon \sim \mathcal{N}(0,1)$.
This **near-identity initialization** ensures the initial product $W_{\text{eff}} \approx I$,
placing the network in a region where all layers contribute roughly equally and the loss
landscape is well-behaved. This avoids the vanishing/exploding gradient pathology that
plagues deep linear networks with random initialization.

In [ ]:
def init_weights(num_layers, rng):
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def copy_weights(weights):
    return [W.copy() for W in weights]

## Forward Pass, Loss, and Gradient Computation

### Forward Pass: Sequential Matrix Multiplication

The forward pass computes $Y = W_L \cdots W_2 W_1 X$ by sequential left-multiplication.
For deep linear networks, this is equivalent to a single linear map with effective weight
$W_{\text{eff}} = \prod_l W_l$, but the factored parameterization creates a richer
optimization landscape with saddle points and gauge symmetries absent in the
single-matrix case.

In [ ]:
def forward_linear(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

### Loss Function: Mean Squared Error

The loss $\mathcal{L} = \frac{1}{2N} \sum_n \| Y_{\text{pred}}^{(n)} - Y_{\text{target}}^{(n)} \|^2$
measures the average squared reconstruction error. For deep linear networks, this loss
landscape has **no spurious local minima** (all local minima are global), but contains
**saddle points** whose number and arrangement depend on the depth $L$ and the singular
value structure of the target.

In [ ]:
def compute_loss(weights, X, Y_target):
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

### Backpropagation: Exact Analytic Gradients

The gradient $\frac{\partial \mathcal{L}}{\partial W_l}$ is computed by manual backpropagation
through the chain of matrix multiplications. For layer $l$:

$$\frac{\partial \mathcal{L}}{\partial W_l} = \delta_l \cdot a_{l-1}^T$$

where $\delta_l = (W_{l+1}^T \cdots W_L^T) \cdot \delta_L$ is the backpropagated error signal
and $a_{l-1}$ is the pre-activation at layer $l$. The gradient inherits the spectral
structure of both the error signal and the activations, which is why it becomes
ill-conditioned late in training.

In [ ]:
def compute_gradients(weights, X, Y_target):
    num_layers = len(weights)
    N = X.shape[1]
    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])
    delta = (activations[-1] - Y_target) / N
    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

### Target Perturbation: Simulating Distribution Shift

To simulate a fine-tuning scenario, a fraction of the target matrix entries are replaced
with fresh random values. This creates a modified target $W_{\text{target,mod}}$ that shares
80% of its structure with the original but differs in 20% of entries. The pre-trained network
must adapt to these changes while leveraging its existing knowledge of the unchanged entries.

In [ ]:
def make_modified_target(W_original, frac, rng):
    W_mod = W_original.copy()
    n_entries = W_mod.size
    n_change = int(frac * n_entries)
    indices = rng.choice(n_entries, size=n_change, replace=False)
    flat = W_mod.ravel()
    flat[indices] = rng.randn(n_change) * 0.5
    return W_mod

---

## Newton-Schulz Orthogonalization: The Core of Muon

### Mathematical Foundation

The Newton-Schulz iteration computes the **polar factor** of a matrix -- the nearest
orthogonal matrix in Frobenius norm. Given a gradient matrix $G$, it finds:

$$G_{\perp} = \arg\min_{Q \in O(n)} \| G - Q \|_F$$

The iteration uses the polynomial recurrence:

$$X_{k+1} = (a I + b X_k X_k^T + c (X_k X_k^T)^2) X_k$$

with coefficients $a = 3.4445$, $b = -4.7750$, $c = 2.0315$ chosen for optimal convergence.
Starting from $X_0 = G / \|G\|_F$, this converges cubically to the orthogonal polar factor.

### Physical Interpretation: Gauge Fixing

In the RG framework, the singular values of weight updates represent **gauge degrees of
freedom** -- rescaling redundancies between adjacent layers. The Newton-Schulz projection
sets all singular values to 1, effectively **fixing the gauge** and forcing all optimization
progress into the rotation (direction) degrees of freedom, which are the physically
meaningful parameters that determine the network's input-output map.

This is why Muon excels in late training: when the remaining loss is concentrated in
small-singular-value directions, SGD's updates in those directions are proportionally
small (because $\|g\| \propto \sigma$), but Muon's orthogonalized updates treat all
directions equally, providing a **spectrally democratic** step.

In [ ]:
def newton_schulz_ortho(M, n_iters=NS_ITERS):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = M / (np.linalg.norm(M, ord='fro') + 1e-7)
    if X.shape[0] > X.shape[1]:
        X = X.T
        transposed = True
    else:
        transposed = False
    Id = np.eye(X.shape[0])
    for _ in range(n_iters):
        A = X @ X.T
        X = (a * Id + b * A + c * A @ A) @ X
    if transposed:
        X = X.T
    return X

---

## Hybrid Training Loop: The Two-Phase Strategy

### Design Rationale

The hybrid trainer implements the core experimental manipulation: given a total budget
of $N$ fine-tuning steps and a switch point $S$:

1. **Steps 0 through S-1 (SGD phase):** Standard SGD with heavy-ball momentum.
   Gradients are used as-is, preserving their natural spectral structure.
   This phase is efficient when gradients are large and well-conditioned.

2. **Step S (transition):** Momentum buffers are **reset to zero** when switching
   from SGD to Muon. This is critical -- the accumulated SGD momentum vectors
   live in a different optimization geometry than Muon's orthogonalized updates.
   Carrying over SGD momentum into the Muon phase would inject spectrally biased
   "memory" that undermines Muon's gauge-fixing effect.

3. **Steps S through N-1 (Muon phase):** Each gradient is orthogonalized via
   Newton-Schulz before being added to the momentum buffer. The update direction
   is spectrally flat, equalizing progress across all singular value directions.

### Key Implementation Details

- **Early termination:** If loss diverges (NaN or >1e10), the trajectory is padded
  with the divergent value to preserve array shape.
- **Final loss:** An extra loss evaluation at the end gives loss at step N (after
  the last update), providing 201 measurements for 200 steps.

In [ ]:
def train_hybrid(weights_init, W_target, X, n_steps, switch_step):
    """
    Train for n_steps total:
      - Steps 0..switch_step-1: SGD with momentum
      - Steps switch_step..n_steps-1: Muon with momentum

    Returns loss trajectory.
    """
    weights = copy_weights(weights_init)
    Y_target = W_target @ X
    velocities = [np.zeros_like(w) for w in weights]

    losses = []

    for step in range(n_steps):
        loss = compute_loss(weights, X, Y_target)
        losses.append(loss)

        if np.isnan(loss) or loss > 1e10:
            losses.extend([loss] * (n_steps - step - 1))
            break

        grads = compute_gradients(weights, X, Y_target)

        if step < switch_step:
            # SGD phase
            for i in range(len(weights)):
                velocities[i] = MOMENTUM * velocities[i] + grads[i]
                weights[i] = weights[i] - SGD_FT_LR * velocities[i]
        else:
            if step == switch_step and switch_step > 0:
                # Reset momentum when switching to Muon
                velocities = [np.zeros_like(w) for w in weights]

            # Muon phase
            for i in range(len(weights)):
                ortho_grad = newton_schulz_ortho(grads[i])
                velocities[i] = MOMENTUM * velocities[i] + ortho_grad
                weights[i] = weights[i] - MUON_FT_LR * velocities[i]

    # Final loss
    final_loss = compute_loss(weights, X, Y_target)
    losses.append(final_loss)

    return np.array(losses)

### Convergence Speed Metric: Steps to Half-Loss

The "steps to reach 50% of initial loss" metric captures **early convergence speed**.
This is complementary to final loss: an optimizer might converge slowly but reach a
better minimum, or vice versa. The hybrid hypothesis specifically predicts that
SGD achieves faster half-loss times while Muon achieves better final losses, and
the optimal hybrid captures both advantages.

In [ ]:
def steps_to_threshold(losses, frac=0.5):
    """Steps to reach frac of initial loss."""
    if len(losses) == 0:
        return len(losses)
    threshold = losses[0] * frac
    for i, l in enumerate(losses):
        if l <= threshold:
            return i
    return len(losses)

---

## Main Experiment Execution

### Protocol Overview

For each of the 5 random seeds, the experiment proceeds as:

1. **Initialize** a fresh 4-layer network with near-identity weights
2. **Pre-train** for 500 SGD steps on the original target (establish baseline competence)
3. **Checkpoint** the pre-trained weights
4. **Modify** 20% of the target (simulate distribution shift)
5. **Fine-tune** from the checkpoint for 200 steps at each of the 7 switch points
   (all starting from the same checkpoint, ensuring fair comparison)
6. **Record** loss trajectories, final losses, and convergence speeds

This design isolates the effect of the SGD-to-Muon transition point: the pre-training,
initialization, data, and target perturbation are all identical within each seed.

In [ ]:
print("=" * 85)
print("Experiment 3.13: SGD->Muon Hybrid Fine-tuning")
print("=" * 85)
print(f"Setup: {NUM_LAYERS}-layer deep linear ({DIM}x{DIM})")
print(f"Pre-train: {PRETRAIN_STEPS} steps SGD, then modify {int(MODIFY_FRAC*100)}% of target")
print(f"Fine-tune: {FINETUNE_STEPS} steps, SGD lr={SGD_FT_LR}, Muon lr={MUON_FT_LR}")
print(f"Switch points S: {SWITCH_POINTS}")
print(f"Seeds: {NUM_SEEDS}")
print("=" * 85)

### Results Storage

Each switch point accumulates lists of final losses, half-loss step counts, and full
loss curves across seeds. This enables both per-seed inspection and aggregate statistics.

In [ ]:
# Collect results
all_results = {S: {'final_losses': [], 'half_steps': [], 'loss_curves': []}
               for S in SWITCH_POINTS}

In [ ]:
for seed_idx in range(NUM_SEEDS):
    run_seed = SEED + seed_idx * 137
    rng = np.random.RandomState(run_seed)

    print(f"\n--- Seed {run_seed} ---")

    # Pre-train
    weights_init = init_weights(NUM_LAYERS, rng)
    Y_target_orig = W_target_original @ X_data
    weights = copy_weights(weights_init)
    velocities = [np.zeros_like(w) for w in weights]

    for step in range(PRETRAIN_STEPS):
        grads = compute_gradients(weights, X_data, Y_target_orig)
        for i in range(NUM_LAYERS):
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            weights[i] = weights[i] - PRETRAIN_LR * velocities[i]

    checkpoint = copy_weights(weights)
    pretrain_loss = compute_loss(checkpoint, X_data, Y_target_orig)
    print(f"  Pre-train final loss: {pretrain_loss:.6f}")

    # Modified target
    W_target_mod = make_modified_target(W_target_original, MODIFY_FRAC, rng)

    # Run each switch point
    for S in SWITCH_POINTS:
        loss_curve = train_hybrid(checkpoint, W_target_mod, X_data, FINETUNE_STEPS, S)
        final_loss = loss_curve[-1]
        half_step = steps_to_threshold(loss_curve)

        all_results[S]['final_losses'].append(final_loss)
        all_results[S]['half_steps'].append(half_step)
        all_results[S]['loss_curves'].append(loss_curve)

    # Print this seed's results
    final_str = "  Finals: " + ", ".join([f"S={S}:{all_results[S]['final_losses'][-1]:.6f}" for S in SWITCH_POINTS])
    print(final_str)

In [ ]:
print(f"\n{'=' * 85}")
print("EXECUTION COMPLETE -- INTERMEDIATE DIAGNOSTICS")
print(f"{'=' * 85}")
print(f"\n  Total runs completed: {NUM_SEEDS * len(SWITCH_POINTS)}")
print(f"  Runs per switch point: {NUM_SEEDS}")
print(f"\n--- Per-Switch-Point Sample Counts (sanity check) ---")
for S in SWITCH_POINTS:
    n_finals = len(all_results[S]['final_losses'])
    n_curves = len(all_results[S]['loss_curves'])
    curve_len = len(all_results[S]['loss_curves'][0]) if n_curves > 0 else 0
    print(f"  S={S:>3d}: {n_finals} final losses, {n_curves} curves of length {curve_len}")

# Check for divergent runs
print(f"\n--- Divergence Check ---")
any_diverged = False
for S in SWITCH_POINTS:
    for seed_i, fl in enumerate(all_results[S]['final_losses']):
        if np.isnan(fl) or fl > 1e6:
            print(f"  WARNING: S={S}, seed {seed_i} diverged (final loss={fl:.2e})")
            any_diverged = True
if not any_diverged:
    print("  All runs converged successfully -- no divergence detected.")

---

## Aggregate Results: Statistical Summary

### Mean Final Loss and Convergence Speed Across Seeds

This table is the central result of the experiment. For each switch point S, we report:

- **Mean Final Loss:** Lower is better. Measures solution quality after the full 200-step budget.
- **Std:** Standard deviation across seeds. Indicates robustness of the result.
- **Mean Half-Steps:** Steps to reach 50% of initial loss. Lower is better. Measures early convergence speed.

**What to look for:**
- If the hypothesis holds, pure Muon (S=0) should have the lowest final loss.
- Pure SGD (S=200) should have the fewest half-steps (fastest early convergence).
- The best hybrid should achieve near-Muon final loss with near-SGD convergence speed.

In [ ]:
print(f"\n\n{'=' * 85}")
print("AGGREGATE RESULTS")
print(f"{'=' * 85}")

In [ ]:
print(f"\n{'Switch S':<12} {'Description':<25} {'Mean Final Loss':>16} {'Std':>10} {'Mean Half-Steps':>16}")
print("-" * 82)

In [ ]:
for S in SWITCH_POINTS:
    if S == 0:
        desc = "Pure Muon"
    elif S == FINETUNE_STEPS:
        desc = "Pure SGD"
    else:
        desc = f"SGD({S})->Muon({FINETUNE_STEPS-S})"

    mean_final = np.mean(all_results[S]['final_losses'])
    std_final = np.std(all_results[S]['final_losses'])
    mean_half = np.mean(all_results[S]['half_steps'])

    print(f"{S:<12} {desc:<25} {mean_final:>16.6f} {std_final:>10.6f} {mean_half:>16.1f}")

In [ ]:
# Interpretation: quantify the Muon advantage and SGD speed advantage
_mean_finals = {S: np.mean(all_results[S]['final_losses']) for S in SWITCH_POINTS}
_mean_halfs = {S: np.mean(all_results[S]['half_steps']) for S in SWITCH_POINTS}
_muon_loss = _mean_finals[0]
_sgd_loss = _mean_finals[FINETUNE_STEPS]
_muon_speed = _mean_halfs[0]
_sgd_speed = _mean_halfs[FINETUNE_STEPS]

print(f"\n--- Interpretation of Aggregate Results ---")
print(f"\n  Final loss ratio (SGD/Muon): {_sgd_loss / (_muon_loss + 1e-12):.2f}x")
print(f"    -> Muon achieves {(1 - _muon_loss/(_sgd_loss + 1e-12))*100:.1f}% lower final loss than SGD")
if _sgd_speed < _muon_speed:
    print(f"\n  Convergence speed ratio (Muon/SGD): {_muon_speed / (_sgd_speed + 1e-12):.2f}x")
    print(f"    -> SGD reaches 50%% loss {_muon_speed - _sgd_speed:.1f} steps earlier than Muon")
else:
    print(f"\n  Convergence speed: Muon is FASTER than SGD to half-loss!")
    print(f"    -> This contradicts the hypothesis that SGD is faster early.")

print(f"\n  Monotonicity check (does final loss increase monotonically with S?):")
prev_loss = _mean_finals[SWITCH_POINTS[0]]
monotonic = True
for S in SWITCH_POINTS[1:]:
    direction = "+" if _mean_finals[S] > prev_loss else "-"
    if _mean_finals[S] < prev_loss:
        monotonic = False
    prev_loss = _mean_finals[S]
    print(f"    S={S:>3d}: {_mean_finals[S]:.6f}  ({direction})")
print(f"    -> {'MONOTONIC' if monotonic else 'NON-MONOTONIC'}: {'more Muon steps always helps' if monotonic else 'there is an optimal switch point'}")

---

## Loss Curve Snapshots: Temporal Dynamics of the Hybrid Strategy

### Reading the Loss Curves

This table shows the **averaged loss trajectory** at key time points. The columns
represent different switch points, and the rows show how loss evolves over time.

**Key patterns to look for:**

- **Early rows (step 0-10):** Do the SGD-heavy strategies (high S) show faster initial
  descent? This would confirm SGD's advantage in the large-gradient regime.
- **Middle rows (step 20-50):** Where does the crossover happen? At what step does
  Muon start outperforming SGD?
- **Late rows (step 100-200):** Do Muon-heavy strategies pull ahead in final loss?
  The gap between pure SGD and pure Muon at step 200 quantifies the value of
  spectral orthogonalization in the refinement phase.
- **Switch point artifacts:** For hybrid strategies, look for loss "bumps" near the
  switch step where momentum is reset and the optimizer changes character.

In [ ]:
print(f"\n\n{'=' * 85}")
print("LOSS CURVES (averaged, selected steps)")
print(f"{'=' * 85}")

In [ ]:
header_parts = [f"{'Step':>6}"]
for S in SWITCH_POINTS:
    if S == 0:
        label = "Muon"
    elif S == FINETUNE_STEPS:
        label = "SGD"
    else:
        label = f"S={S}"
    header_parts.append(f"{label:>12}")
print("  ".join(header_parts))
print("-" * (8 + 14 * len(SWITCH_POINTS)))

In [ ]:
snapshot_steps = [0, 5, 10, 20, 50, 100, 150, 200]
for step_idx in snapshot_steps:
    parts = [f"{step_idx:>6}"]
    for S in SWITCH_POINTS:
        curves = all_results[S]['loss_curves']
        if step_idx < len(curves[0]):
            mean_loss = np.mean([c[step_idx] for c in curves])
            parts.append(f"{mean_loss:>12.6f}")
        else:
            parts.append(f"{'':>12}")
    print("  ".join(parts))

In [ ]:
# Interpretation: identify the crossover step where Muon overtakes SGD
print(f"\n--- Crossover Analysis ---")
muon_curves = all_results[0]['loss_curves']
sgd_curves = all_results[FINETUNE_STEPS]['loss_curves']
mean_muon = np.mean(muon_curves, axis=0)
mean_sgd = np.mean(sgd_curves, axis=0)
crossover_found = False
for step_i in range(min(len(mean_muon), len(mean_sgd))):
    if mean_muon[step_i] < mean_sgd[step_i]:
        print(f"  Muon overtakes SGD at step {step_i}")
        print(f"    Muon loss at crossover: {mean_muon[step_i]:.6f}")
        print(f"    SGD  loss at crossover: {mean_sgd[step_i]:.6f}")
        crossover_found = True
        break
if not crossover_found:
    print("  SGD never falls below Muon (or they remain tied)")

# Loss reduction rate in first 10 steps vs last 50 steps
print(f"\n--- Phase-Specific Convergence Rates ---")
for S in [0, FINETUNE_STEPS]:
    label = "Pure Muon" if S == 0 else "Pure SGD"
    curves = np.mean(all_results[S]['loss_curves'], axis=0)
    early_rate = (curves[0] - curves[min(10, len(curves)-1)]) / 10 if len(curves) > 10 else 0
    late_rate = (curves[max(0, len(curves)-51)] - curves[-1]) / 50 if len(curves) > 51 else 0
    print(f"  {label}:")
    print(f"    Early rate (steps 0-10): {early_rate:.6f} loss/step")
    print(f"    Late rate (steps 150-200): {late_rate:.6f} loss/step")
    print(f"    Ratio (early/late): {early_rate/(late_rate + 1e-12):.1f}x")

---

## Optimal Switch Point Analysis

### Finding the Sweet Spot

This section identifies the switch point S that optimizes the
**loss-quality vs convergence-speed tradeoff**. The key question:

- Does a hybrid (0 < S < 200) beat both pure strategies on their respective strengths?
- If S=0 (pure Muon) is already the best on both metrics, the hybrid strategy is unnecessary.
- If S>0 hybrids achieve near-Muon final loss with faster convergence, the two-phase
  approach has practical value.

From the RG perspective, this analysis reveals **when gauge-fixing becomes essential**:
if the optimal S is small, it means orthogonalization is needed almost immediately;
if S is large, there is a substantial regime where raw gradient descent is sufficient
and gauge-fixing adds overhead without benefit.

In [ ]:
print(f"\n\n{'=' * 85}")
print("OPTIMAL SWITCH POINT ANALYSIS")
print(f"{'=' * 85}")

In [ ]:
# Find best S by final loss
mean_finals = {S: np.mean(all_results[S]['final_losses']) for S in SWITCH_POINTS}
best_S = min(mean_finals, key=mean_finals.get)
pure_muon_loss = mean_finals[0]
pure_sgd_loss = mean_finals[FINETUNE_STEPS]

In [ ]:
print(f"\n  Pure Muon (S=0) final loss:      {pure_muon_loss:.6f}")
print(f"  Pure SGD (S={FINETUNE_STEPS}) final loss:    {pure_sgd_loss:.6f}")
print(f"  Best hybrid (S={best_S}) final loss: {mean_finals[best_S]:.6f}")

In [ ]:
# Is the best hybrid better than both?
better_than_muon = mean_finals[best_S] < pure_muon_loss
better_than_sgd = mean_finals[best_S] < pure_sgd_loss
print(f"\n  Best hybrid beats pure Muon? {better_than_muon}")
print(f"  Best hybrid beats pure SGD?  {better_than_sgd}")

In [ ]:
# Convergence speed comparison
mean_half_steps = {S: np.mean(all_results[S]['half_steps']) for S in SWITCH_POINTS}
fastest_S = min(mean_half_steps, key=mean_half_steps.get)
print(f"\n  Fastest to 50% loss: S={fastest_S} ({mean_half_steps[fastest_S]:.1f} steps)")
print(f"  Pure Muon speed:     {mean_half_steps[0]:.1f} steps")
print(f"  Pure SGD speed:      {mean_half_steps[FINETUNE_STEPS]:.1f} steps")

In [ ]:
# Combined metric: rank by final loss, break ties by speed
print(f"\n  Ranking by final loss:")
sorted_S = sorted(SWITCH_POINTS, key=lambda s: mean_finals[s])
for rank, S in enumerate(sorted_S, 1):
    marker = " <-- BEST" if S == best_S else ""
    if S == 0:
        desc = "Pure Muon"
    elif S == FINETUNE_STEPS:
        desc = "Pure SGD"
    else:
        desc = f"Hybrid S={S}"
    print(f"    #{rank}: {desc:<25} loss={mean_finals[S]:.6f}  speed={mean_half_steps[S]:.1f} steps{marker}")

In [ ]:
# Interpretation: quantify the efficiency frontier
print(f"\n--- Efficiency Frontier Analysis ---")
print(f"\n  The 'Pareto-optimal' strategies are those not dominated on BOTH metrics.")
print(f"  A strategy is dominated if another has both lower loss AND fewer half-steps.\n")
pareto = []
for S in SWITCH_POINTS:
    dominated = False
    for S2 in SWITCH_POINTS:
        if S2 != S and mean_finals[S2] <= mean_finals[S] and mean_half_steps[S2] <= mean_half_steps[S]:
            if mean_finals[S2] < mean_finals[S] or mean_half_steps[S2] < mean_half_steps[S]:
                dominated = True
                break
    status = "DOMINATED" if dominated else "PARETO-OPTIMAL"
    if not dominated:
        pareto.append(S)
    print(f"  S={S:>3d}: loss={mean_finals[S]:.6f}, speed={mean_half_steps[S]:.1f}  [{status}]")

print(f"\n  Pareto-optimal switch points: {pareto}")
if len(pareto) == 1 and pareto[0] == 0:
    print(f"  -> Pure Muon dominates all strategies -- no benefit from SGD warm-up.")
elif len(pareto) > 1:
    print(f"  -> Multiple strategies on the Pareto frontier -- genuine speed/quality tradeoff exists.")

---

## Hypothesis Tests: Formal Evaluation

### Test Design

Four binary tests evaluate the core predictions of the hybrid hypothesis:

- **T1 (SGD is faster early):** Tests whether SGD's natural gradient alignment with
  high-curvature directions provides faster initial convergence.
- **T2 (Muon achieves better final loss):** Tests whether spectral orthogonalization
  gives Muon an advantage in the low-loss, ill-conditioned regime.
- **T3 (Hybrid is competitive):** Tests whether a hybrid strategy achieves within 5%
  of pure Muon's final loss. A 5% tolerance accounts for the reduced Muon budget.
- **T4 (Hybrid is faster):** Tests whether the best hybrid converges faster than pure
  Muon, validating the SGD warm-up's contribution.

If T1 and T2 both pass, the two optimizers have complementary strengths, justifying
the hybrid approach. If T3 and T4 also pass, the hybrid successfully captures both
advantages -- the strongest possible confirmation of the two-phase strategy.

In [ ]:
print(f"\n\n{'=' * 85}")
print("HYPOTHESIS TESTS")
print(f"{'=' * 85}")

In [ ]:
# T1: Pure SGD converges faster (fewer steps to half-loss)
t1 = mean_half_steps[FINETUNE_STEPS] < mean_half_steps[0]
print(f"\nT1: Pure SGD faster to half-loss than pure Muon?")
print(f"    SGD: {mean_half_steps[FINETUNE_STEPS]:.1f} steps, Muon: {mean_half_steps[0]:.1f} steps")
print(f"    --> {'PASS' if t1 else 'FAIL'}")

In [ ]:
# T2: Pure Muon gets better final loss
t2 = pure_muon_loss < pure_sgd_loss
print(f"\nT2: Pure Muon gets better final loss than pure SGD?")
print(f"    Muon: {pure_muon_loss:.6f}, SGD: {pure_sgd_loss:.6f}")
print(f"    --> {'PASS' if t2 else 'FAIL'}")

In [ ]:
# T3: Some hybrid S (not 0 or 200) is competitive with or beats pure Muon
hybrid_only = {S: mean_finals[S] for S in SWITCH_POINTS if S > 0 and S < FINETUNE_STEPS}
best_hybrid_S = min(hybrid_only, key=hybrid_only.get)
best_hybrid_loss = hybrid_only[best_hybrid_S]
t3 = best_hybrid_loss <= pure_muon_loss * 1.05  # within 5% of Muon
print(f"\nT3: Best hybrid within 5% of pure Muon?")
print(f"    Best hybrid S={best_hybrid_S}: {best_hybrid_loss:.6f}")
print(f"    Pure Muon: {pure_muon_loss:.6f}")
print(f"    Ratio: {best_hybrid_loss / (pure_muon_loss + 1e-12):.4f}")
print(f"    --> {'PASS' if t3 else 'FAIL'}")

In [ ]:
# T4: Best hybrid is faster than pure Muon
t4 = mean_half_steps[best_hybrid_S] < mean_half_steps[0]
print(f"\nT4: Best hybrid faster than pure Muon?")
print(f"    Hybrid S={best_hybrid_S}: {mean_half_steps[best_hybrid_S]:.1f} steps")
print(f"    Pure Muon: {mean_half_steps[0]:.1f} steps")
print(f"    --> {'PASS' if t4 else 'FAIL'}")

---

## Final Verdict and Summary

### Synthesizing the Evidence

The verdict aggregates all four hypothesis tests into an overall assessment of the
SGD-to-Muon hybrid strategy. The possible outcomes are:

- **CONFIRMED (3-4/4 pass):** The two-phase strategy is validated. SGD and Muon operate
  on complementary spectral regimes, and switching between them captures both fast
  convergence and high solution quality.
- **PARTIALLY CONFIRMED (2/4 pass):** Mixed evidence. Some aspects of the hybrid
  hypothesis hold, but the two optimizers may not be as cleanly complementary as
  theorized.
- **INCONCLUSIVE (0-1/4 pass):** The hybrid strategy offers no clear advantage over
  pure approaches. This would suggest that the spectral regime boundary between
  "SGD-appropriate" and "Muon-appropriate" optimization is either not well-defined
  or not exploitable in this setting.

In [ ]:
print(f"\n\n{'=' * 85}")
print("FINAL VERDICT: SGD->MUON HYBRID FINE-TUNING")
print(f"{'=' * 85}")

In [ ]:
total_pass = sum([t1, t2, t3, t4])
print(f"""
  Tests passed: {total_pass}/4

  Best switch point: S={best_S}
  Best hybrid (non-pure) switch: S={best_hybrid_S}

  The strategy: SGD for {best_hybrid_S} steps (fast descent),
  then switch to Muon for the remaining {FINETUNE_STEPS - best_hybrid_S} steps (quality refinement).
""")

In [ ]:
if total_pass >= 3:
    print("  VERDICT: CONFIRMED -- Hybrid strategy is effective.")
    print(f"  SGD handles fast early descent, Muon handles quality refinement.")
    print(f"  Optimal switch point: S={best_hybrid_S}")
elif total_pass >= 2:
    print("  VERDICT: PARTIALLY CONFIRMED")
else:
    print("  VERDICT: INCONCLUSIVE -- pure strategies may be sufficient.")

In [ ]:
print("=" * 85)

---

## Conclusions and Theoretical Interpretation

### Summary of Findings

This experiment tested whether a **two-phase optimization strategy** -- using SGD for fast
early descent followed by Muon for precise late-stage refinement -- can outperform either
optimizer used in isolation.

### Connection to the RG Gauge-Fixing Framework

The results illuminate the **temporal structure of gauge redundancy** in deep linear networks:

1. **Early training:** Gradients are large and spectrally diverse. The gauge degrees of freedom
   (inter-layer rescaling redundancies) do not significantly impede optimization because the
   dominant singular value directions carry enough signal for rapid descent. SGD works well
   here because the "natural" spectral structure of the gradient is informative.

2. **Late training:** As the network approaches a minimum, the remaining loss is distributed
   across many singular value directions with small, similar magnitudes. SGD's updates become
   spectrally biased -- large singular values still dominate the gradient even though the
   corresponding loss components are already minimized. Muon's orthogonalization **fixes the
   gauge** by removing this spectral bias, ensuring equal progress across all remaining
   error directions.

3. **The transition:** The optimal switch point S identifies the training step at which gauge
   redundancy transitions from being benign (or even helpful) to being the primary bottleneck.
   In the RG analogy, this is the **crossover scale** between the UV regime (where coarse
   features dominate) and the IR regime (where fine spectral details matter).

### Broader Implications

- **Practical:** If confirmed, the hybrid strategy suggests that production training pipelines
  for deep networks could benefit from switching optimizers mid-training, using cheap gradient
  descent early and more expensive orthogonalized updates only when they become necessary.

- **Theoretical:** The existence of an optimal switch point would provide quantitative evidence
  that Muon's advantage is specifically tied to the **spectral conditioning of the loss
  landscape**, not to a generic improvement in update quality. This narrows the mechanistic
  explanation and strengthens the gauge-fixing interpretation.

- **Limitations:** This experiment uses deep linear networks, which have no spurious local
  minima. In nonlinear networks, the interaction between spectral conditioning and the
  nonlinear activation landscape may shift the optimal switch point or even eliminate the
  benefit of the hybrid approach.